# Module 1: Business Understanding & Synthetic Data Generation
**Real-Time Payment Fraud Detection for an E-Commerce Platform**

| Section | Focus Area | Alignment with Project Documents |
|---|---|---|
| **1. Problem Formulation & Business KPIs** | Define core business problem, operational guardrails, and baseline policies | Matches `Business-Understanding-EDA-14-10.pdf` (Slides 2–14, 35) & `EDA-E-commerce_Fraud_Detection.ipynb` (`[S0]`, `[Q1.1]`, `[Q4.1]`, `[Q4.2]`) |
| **2. Rationale for Synthetic Extension** | Initiation-time signal limitation, contextual telemetry extension, and KPI alignment | Matches verbatim Section 1 of `[BA] Synthetic Data Strategy & Data Dictionary.docx` |
| **3. Data Dictionary** | Exact specification of the 6 synthetically generated fields | Matches verbatim Section 2 of `[BA] Synthetic Data Strategy & Data Dictionary.docx` |
| **4. Synthetic Data Generation** | Vectorized Python generation on the PaySim base log (`6,362,620` rows) | Strictly implements the exact probability distributions and parameters in the Data Dictionary |

In [ ]:
# ===== [S0] Setup: libraries, configuration, number formatting, helpers (Matched with EDA Notebook) =====
import os, textwrap, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullFormatter
from IPython.display import display
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
np.random.seed(42)

# ---------- Configuration (Exact match with EDA-E-commerce_Fraud_Detection.ipynb) ----------
SAMPLE_NROWS    = None        # None = WHOLE dataset (6,362,620 rows). Use 100 only to check syntax.
DATA_PATH       = "/kaggle/input/datasets/rupakroy/online-payments-fraud-detection-dataset/PS_20174392719_1491204439457_log.csv"
FLAG_THRESHOLD  = 200_000     # documented rule (dataset documentation): an attempt to TRANSFER more than 200,000 in one transaction
MODEL_TYPES     = ["TRANSFER", "CASH_OUT"]   # types in which fraud occurs (verified in [Q2.2])
FPR_GUARDRAIL   = 0.0151      # framing-document reference: global e-commerce false-positive rate (1.51%), measured BY VALUE
ADDITIONAL_COST = 0.0         # KPI terms "fee or margin lost on false declines" and "review cost": not recorded in PaySim -> 0 by design
K_VALUES        = [16, 100, 1_000]  # Precision@k (K = 16 alerts of current rule, 100, 1000)
SPLIT_STEPS     = (504, 624)  # PROPOSED time-ordered windows: train = steps 1-504 (days 1-21), validation = 505-624 (days 22-26), test = 625-end (days 27-31)

OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./working"
for sub in ("tables", "figures"):
    os.makedirs(f"{OUT_DIR}/{sub}", exist_ok=True)

# ---------- Number formatting: never scientific notation ----------
np.set_printoptions(suppress=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

def sdiv(a, b):
    """Safe division: NaN instead of an error when the denominator is 0."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    r = np.divide(a, b, out=np.full(np.broadcast(a, b).shape, np.nan), where=(b != 0))
    return float(r) if r.ndim == 0 else r

def _fmt_col(s, decimals, as_pct):
    if as_pct:
        return s.map(lambda v: "" if pd.isna(v) else f"{v:,.2f}%")
    if pd.api.types.is_bool_dtype(s) or not pd.api.types.is_numeric_dtype(s):
        return s.astype(str)
    if pd.api.types.is_integer_dtype(s):
        return s.map(lambda v: f"{v:,}")
    whole = bool(s.dropna().mod(1).eq(0).all() and s.abs().max() < 1e15)
    return s.map(lambda v: "" if pd.isna(v) else (f"{v:,.0f}" if whole else f"{v:,.{decimals}f}"))

def format_table(df, decimals=2, pct=None):
    cols = [c for c in df.columns if str(c).endswith("_pct")] if pct is None else (list(df.columns) if pct == "all" else pct)
    return pd.DataFrame({c: _fmt_col(df[c], decimals, c in cols) for c in df.columns}, index=df.index)

def show(df, name, decimals=2, pct=None, transpose=False, index=True):
    f = format_table(df, decimals, pct)
    f = f.T if transpose else f
    display(f if index else f.reset_index(drop=True))
    df.to_csv(f"{OUT_DIR}/tables/{name}.csv", index=index)

## 1. Problem Definition, Business KPIs & Policy Baselines
### 1.1 Core Problem Statement & Operational Objective
* **Core Problem Statement**: *"Given the information available when a transaction is initiated (type, amount, time step and account identifiers), which transactions are likely to be fraudulent, and which action minimises the total fraud cost, subject to limits on customer friction and review capacity?"*
* **Objective**: Minimize the net fraud cost rate subject to the false-positive rate (by value) and the review load (alerts per step) remaining within agreed guardrails.
* **Operational Objective**: Maximise the value-weighted detection rate (equivalently, minimise net fraud cost rate) subject to the false-positive rate by value staying within the **1.51%** reference (NHIMG 2026).
* **PaySim Approximations & Double Counting Note**:
  * Additional-cost terms (fee/margin lost on false declines, manual-review cost) are not recorded in PaySim and are set to `0` by design: $\text{Net fraud cost rate} = \text{Fraud value not stopped} / \text{TPV}$, and $\text{Net financial loss avoided} = \text{Fraud value stopped}$.
  * In PaySim's account takeover scenario, the same funds appear twice (as a fraudulent `TRANSFER` and a fraudulent `CASH_OUT`). Headline value-based KPIs sum all fraudulent rows (both legs), while a sensitivity analysis counts only the `TRANSFER` leg as funds actually drained.

In [ ]:
# ===== [M1.1] Load PaySim Base Dataset & Verify Integrity (Exact match with [Q1.1]) =====
TYPES = ["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"]
EXPECTED = ["step", "type", "amount", "nameOrig", "oldbalanceOrg", "newbalanceOrig",
            "nameDest", "oldbalanceDest", "newbalanceDest", "isFraud", "isFlaggedFraud"]

raw = pd.read_csv(DATA_PATH, nrows=SAMPLE_NROWS)
assert list(raw.columns) == EXPECTED, f"Unexpected columns: {list(raw.columns)}"
raw["type"] = raw["type"].astype(str).str.strip().str.upper().str.replace("-", "_", regex=False)
assert raw["type"].isin(TYPES).all(), "Unknown transaction type present"
if not raw["step"].is_monotonic_increasing:
    raw = raw.sort_values("step", kind="stable").reset_index(drop=True)

N = len(raw)
AMT = raw["amount"].to_numpy()
Y = raw["isFraud"].to_numpy().astype(bool)
TOTAL_FRAUD_N = int(Y.sum())
TOTAL_FRAUD_V = float(AMT[Y].sum())
LEGIT_N = int((~Y).sum())
LEGIT_V = float(AMT[~Y].sum())
TPV = float(AMT.sum())
N_STEPS_OBS = int(raw["step"].nunique())

snapshot = pd.DataFrame({
    "rows": [N],
    "fraud_cases": [TOTAL_FRAUD_N],
    "fraud_rate_pct": [100 * sdiv(TOTAL_FRAUD_N, N)],
    "fraud_value": [TOTAL_FRAUD_V],
    "total_payment_value": [TPV],
    "fraud_loss_rate_pct": [100 * sdiv(TOTAL_FRAUD_V, TPV)]
}, index=["PaySim base log"])
show(snapshot, "M1_1_base_dataset_snapshot")

In [ ]:
# ===== [M1.2] Business KPIs & Policy Baselines Evaluation (Matches Slide 11 & Slide 35) =====
is_transfer = (raw["type"] == "TRANSFER").to_numpy()
in_scope = raw["type"].isin(MODEL_TYPES).to_numpy()

# 1. Policy flags
flag_do_nothing = np.zeros(N, dtype=bool)
flag_current_rule = raw["isFlaggedFraud"].to_numpy().astype(bool)
flag_documented_rule = is_transfer & (AMT > FLAG_THRESHOLD)

# Amount-only threshold calibrated to 1.51% FPR by value on legitimate transactions
legit_sc_amt = np.sort(AMT[(~Y) & in_scope])[::-1]
cum_legit_v = np.cumsum(legit_sc_amt)
idx_guardrail = np.searchsorted(cum_legit_v, FPR_GUARDRAIL * LEGIT_V, side="right") - 1
amt_threshold_guardrail = float(legit_sc_amt[max(idx_guardrail, 0)]) if legit_sc_amt.size else np.inf
flag_amount_guardrail = in_scope & (AMT >= amt_threshold_guardrail)

def evaluate_policy(flag_mask, policy_name):
    alerts = int(flag_mask.sum())
    tp_n = int((flag_mask & Y).sum())
    fp_n = int((flag_mask & (~Y)).sum())
    
    caught_fraud_v = float(AMT[flag_mask & Y].sum())
    missed_fraud_v = TOTAL_FRAUD_V - caught_fraud_v
    false_decline_v = float(AMT[flag_mask & (~Y)].sum())
    
    return {
        "Policy": policy_name,
        "Value_Detect_pct": 100 * sdiv(caught_fraud_v, TOTAL_FRAUD_V),
        "False_Decline_pct": 100 * sdiv(fp_n, LEGIT_N),
        "FPR_Val_pct": 100 * sdiv(false_decline_v, LEGIT_V),
        "Alert_Prec_pct": 100 * sdiv(tp_n, alerts),
        "Net_Cost_Rate_pct": 100 * sdiv(missed_fraud_v + ADDITIONAL_COST, TPV),
        "Net_Loss_Avoided": caught_fraud_v - ADDITIONAL_COST,
        "Alerts_per_Step": sdiv(alerts, N_STEPS_OBS)
    }

policies_df = pd.DataFrame([
    evaluate_policy(flag_do_nothing, "Do nothing"),
    evaluate_policy(flag_current_rule, "Current rule (isFlaggedFraud)"),
    evaluate_policy(flag_documented_rule, "Documented rule (TRANSFER > 200k)"),
    evaluate_policy(flag_amount_guardrail, f"Amount-only (Guardrail 1.51%, >= {amt_threshold_guardrail:,.2f})")
]).set_index("Policy")

show(policies_df, "M1_2_policy_baselines_kpi_benchmark")

## 2. Rationale for Synthetic Extension & Complete Data Dictionary
*(Verbatim from `[BA] Synthetic Data Strategy & Data Dictionary.docx`)*

### 2.1 Rationale for Synthetic Extension
* **Initiation-time signal limitation**: The four balance columns (`oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`, `newbalanceDest`) reflect post-decision cancellation states and serve strictly diagnostic roles. Excluding them leaves only basic transactional fields (`step`, `type`, `amount`, and account IDs) at initiation time, which lack the granularity needed to separate high-value legitimate transactions from fraud within the 1.51% false-positive rate guardrail.
* **Contextual telemetry extension**: The base PaySim log is augmented with six synthetic contextual and behavioral attributes—capturing account tenure, device trust, IP-to-billing distance, address consistency, and prior authentication failures.
* **Alignment with fraud scenario and KPIs**: Designed around the Account Takeover pattern (`TRANSFER` followed by `CASH_OUT`), these initiation-time signals establish a continuous risk space required for `Precision@K` ($K=16, 100, 1000$) ranking and downstream cost-sensitive threshold optimization.

In [ ]:
# ===== [M1.3] Produce Complete Data Dictionary (Exact Verbatim Match with DOCX) =====
data_dictionary_df = pd.DataFrame([
    {
        "Column Name": "account_age_days",
        "Data Type": "Integer",
        "Unit / Format": "Days",
        "Valid Range": "[1, 3650]",
        "Business Assumption & Generation Logic": (
            "Assumption: Newly created or recently registered throwaway accounts carry higher fraud risk, "
            "whereas long-tenured accounts are predominantly legitimate. "
            "Logic: isFraud = 0: Drawn from a Gamma distribution skewed toward mature accounts (shape = 3.0, scale = 150.0, clipped at [1, 3650]). "
            "isFraud = 1: Mixture distribution where 70% are young accounts (Exponential with scale = 25 days) and 30% are mature accounts (representing Account Takeover of established users)."
        )
    },
    {
        "Column Name": "device_trust_score",
        "Data Type": "Float",
        "Unit / Format": "Score",
        "Valid Range": "[0.0, 1.0]",
        "Business Assumption & Generation Logic": (
            "Assumption: Device/browser fingerprinting measures whether the transaction originates from a recognized, trusted hardware/browser history. "
            "Logic: isFraud = 0: Sampled from Beta(alpha = 8, beta = 2), concentrated near 0.70-1.00 with a small tail for legitimate users on new devices. "
            "isFraud = 1: Sampled from Beta(alpha = 2, beta = 6), concentrated near 0.05-0.45, reflecting unrecognized devices or emulators used by attackers."
        )
    },
    {
        "Column Name": "is_new_device",
        "Data Type": "Integer (Binary)",
        "Unit / Format": "Flag",
        "Valid Range": "{0, 1}",
        "Business Assumption & Generation Logic": (
            "Assumption: Account takeover typically occurs from a device never previously associated with the user's account. "
            "Logic: Bernoulli trial with P(new_device = 1) = 0.08 for isFraud = 0 (normal device upgrades/secondary devices) vs. P(new_device = 1) = 0.72 for isFraud = 1."
        )
    },
    {
        "Column Name": "ip_billing_distance_km",
        "Data Type": "Float",
        "Unit / Format": "Kilometers (km)",
        "Valid Range": "[0.0, 15000.0]",
        "Business Assumption & Generation Logic": (
            "Assumption: A large physical distance between the IP geolocation at transaction initiation and the registered billing address indicates potential VPN/proxy usage or foreign account takeover. "
            "Logic: isFraud = 0: Log-normal distribution centered around local proximity (mu = 2.5, sigma = 1.2), mixed with 3% uniform outliers [500, 5000] km representing legitimate travelers. "
            "isFraud = 1: Log-normal distribution (mu = 6.2, sigma = 1.5) shifted toward long-distance/cross-border anomalies, clipped at 15,000 km."
        )
    },
    {
        "Column Name": "shipping_billing_mismatch",
        "Data Type": "Integer (Binary)",
        "Unit / Format": "Flag",
        "Valid Range": "{0, 1}",
        "Business Assumption & Generation Logic": (
            "Assumption: In e-commerce payments (PAYMENT) or account-takeover fund transfers (TRANSFER / CASH_OUT), fraudsters frequently route goods or cash-out funds to a destination/beneficiary registered region that mismatches the originating account's primary billing address. "
            "Logic: Bernoulli trial with P(mismatch = 1) = 0.05 for isFraud = 0 (e.g., legitimate customers sending gifts) vs. P(mismatch = 1) = 0.60 for isFraud = 1."
        )
    },
    {
        "Column Name": "failed_auth_attempts_24h",
        "Data Type": "Integer",
        "Unit / Format": "Count",
        "Valid Range": "[0, 15]",
        "Business Assumption & Generation Logic": (
            "Assumption: Brute-force login attempts, card testing, or repeated PIN/OTP failures in the 24 hours prior to a transaction strongly signal unauthorized access. "
            "Logic: isFraud = 0: Poisson distribution (lambda = 0.15, ~86% have 0 failed attempts), clipped at 15. "
            "isFraud = 1: Poisson distribution (lambda = 2.8), clipped at 15, reflecting credential stuffing or repeated trial-and-error attempts."
        )
    }
]).set_index("Column Name")

show(data_dictionary_df, "M1_3_synthetic_data_dictionary")

## 3. Synthetic Contextual & Behavioral Data Generation in Python
Strictly implements the exact distributions and parameters documented in Section 2 of `[BA] Synthetic Data Strategy & Data Dictionary.docx`.

In [ ]:
# ===== [M1.4] Generate the 6 Synthetic Contextual Fields on PaySim Dataset =====
def generate_synthetic_extension(df: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    """
    Generates the 6 contextual telemetry fields strictly following the Data Dictionary specifications.
    """
    rng = np.random.default_rng(seed)
    n_rows = len(df)
    is_fraud = df["isFraud"].to_numpy().astype(bool)
    n_fraud = int(is_fraud.sum())
    n_legit = n_rows - n_fraud

    # 1. account_age_days: Integer, [1, 3650]
    # isFraud = 0: Gamma(shape = 3.0, scale = 150.0), clipped at [1, 3650]
    # isFraud = 1: Mixture 70% Exponential(scale = 25) and 30% Gamma(shape = 3.0, scale = 150.0), clipped at [1, 3650]
    age_legit = rng.gamma(shape=3.0, scale=150.0, size=n_legit)
    
    young_mask = rng.random(n_fraud) < 0.70
    age_fraud = np.empty(n_fraud, dtype=np.float64)
    age_fraud[young_mask] = rng.exponential(scale=25.0, size=int(young_mask.sum()))
    age_fraud[~young_mask] = rng.gamma(shape=3.0, scale=150.0, size=int((~young_mask).sum()))
    
    account_age_days = np.empty(n_rows, dtype=np.int32)
    account_age_days[~is_fraud] = np.clip(np.rint(age_legit), 1, 3650).astype(np.int32)
    account_age_days[is_fraud] = np.clip(np.rint(age_fraud), 1, 3650).astype(np.int32)
    df["account_age_days"] = account_age_days

    # 2. device_trust_score: Float, [0.0, 1.0]
    # isFraud = 0: Beta(alpha = 8, beta = 2)
    # isFraud = 1: Beta(alpha = 2, beta = 6)
    device_trust_score = np.empty(n_rows, dtype=np.float32)
    device_trust_score[~is_fraud] = rng.beta(a=8.0, b=2.0, size=n_legit).astype(np.float32)
    device_trust_score[is_fraud] = rng.beta(a=2.0, b=6.0, size=n_fraud).astype(np.float32)
    df["device_trust_score"] = np.clip(device_trust_score, 0.0, 1.0)

    # 3. is_new_device: Integer (Binary), {0, 1}
    # isFraud = 0: Bernoulli(p = 0.08)
    # isFraud = 1: Bernoulli(p = 0.72)
    is_new_device = np.empty(n_rows, dtype=np.int8)
    is_new_device[~is_fraud] = (rng.random(n_legit) < 0.08).astype(np.int8)
    is_new_device[is_fraud] = (rng.random(n_fraud) < 0.72).astype(np.int8)
    df["is_new_device"] = is_new_device

    # 4. ip_billing_distance_km: Float, [0.0, 15000.0]
    # isFraud = 0: Log-normal(mu = 2.5, sigma = 1.2) mixed with 3% Uniform[500, 5000]
    # isFraud = 1: Log-normal(mu = 6.2, sigma = 1.5), clipped at 15,000 km
    dist_legit = rng.lognormal(mean=2.5, sigma=1.2, size=n_legit)
    outlier_mask = rng.random(n_legit) < 0.03
    dist_legit[outlier_mask] = rng.uniform(low=500.0, high=5000.0, size=int(outlier_mask.sum()))
    
    dist_fraud = rng.lognormal(mean=6.2, sigma=1.5, size=n_fraud)
    
    ip_billing_distance_km = np.empty(n_rows, dtype=np.float32)
    ip_billing_distance_km[~is_fraud] = np.clip(dist_legit, 0.0, 15000.0).astype(np.float32)
    ip_billing_distance_km[is_fraud] = np.clip(dist_fraud, 0.0, 15000.0).astype(np.float32)
    df["ip_billing_distance_km"] = ip_billing_distance_km

    # 5. shipping_billing_mismatch: Integer (Binary), {0, 1}
    # isFraud = 0: Bernoulli(p = 0.05)
    # isFraud = 1: Bernoulli(p = 0.60)
    shipping_billing_mismatch = np.empty(n_rows, dtype=np.int8)
    shipping_billing_mismatch[~is_fraud] = (rng.random(n_legit) < 0.05).astype(np.int8)
    shipping_billing_mismatch[is_fraud] = (rng.random(n_fraud) < 0.60).astype(np.int8)
    df["shipping_billing_mismatch"] = shipping_billing_mismatch

    # 6. failed_auth_attempts_24h: Integer, [0, 15]
    # isFraud = 0: Poisson(lambda = 0.15), clipped at 15
    # isFraud = 1: Poisson(lambda = 2.8), clipped at 15
    failed_auth = np.empty(n_rows, dtype=np.int8)
    failed_auth[~is_fraud] = np.clip(rng.poisson(lam=0.15, size=n_legit), 0, 15).astype(np.int8)
    failed_auth[is_fraud] = np.clip(rng.poisson(lam=2.80, size=n_fraud), 0, 15).astype(np.int8)
    df["failed_auth_attempts_24h"] = failed_auth

    return df

raw = generate_synthetic_extension(raw, seed=42)
display(raw.head(10))

In [ ]:
# ===== [M1.5] Verification of Generated Fields (Range, Summary Statistics by Class & Single-Feature AUC) =====
SYNTH_COLS = [
    "account_age_days", "device_trust_score", "is_new_device",
    "ip_billing_distance_km", "shipping_billing_mismatch", "failed_auth_attempts_24h"
]

verification_rows = []
for col in SYNTH_COLS:
    s_all = raw[col]
    s0 = raw.loc[~Y, col]
    s1 = raw.loc[Y, col]
    auc_val = roc_auc_score(Y, s_all) if 0 < Y.sum() < len(Y) else np.nan
    verification_rows.append({
        "column": col,
        "dtype": str(s_all.dtype),
        "null_cells": int(s_all.isna().sum()),
        "min": float(s_all.min()),
        "max": float(s_all.max()),
        "legit_mean": float(s0.mean()),
        "legit_median": float(s0.median()),
        "fraud_mean": float(s1.mean()),
        "fraud_median": float(s1.median()),
        "single_feature_auc": auc_val,
        "separation_abs_2auc_minus_1": abs(2 * auc_val - 1)
    })

ver_df = pd.DataFrame(verification_rows).set_index("column")
show(ver_df, "M1_5_synthetic_fields_verification", decimals=4)

# Save augmented dataset for downstream modules
augmented_path = f"{OUT_DIR}/paysim_augmented_module1.parquet"
raw.to_parquet(augmented_path, index=False)
print(f"Saved augmented dataset to: {augmented_path}")